<a href="https://colab.research.google.com/github/menychtak/diffusion-face-privacy-emotion/blob/main/notebooks/24_unified_heldout_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 24 — Unified held-out evaluation (NB23)

**Run all in Colab after NB23 has 324/324 verified outputs.** No diffusion generation, no old notebooks rerun, no tuning-source reuse. Final three conditions × 108 held-out source faces × seed 42. Frozen N12 V3 SCRFD alignment + ArcFace/AuraFace, original N07 pinned two-ViT softmax ensemble, and N16 pyiqa NIQE 0.1.13.

**Scientific safeguards:** fail-closed source replay against N07 FER baseline; verified SHA receipts; exact detector/recognizer weight hashes; never zero-fill failed biometric detection; mean source-cluster/bootstrap comparisons and no re-identification-rate claim. Source-level QC remains pending until manually reviewed. Private per-image evidence goes only to Drive; **do not commit rows/images publicly.**

**Runtime:** Colab Python 3.12; CPU is sufficient, GPU speeds FER. NIQE uses CPU. The environment cell may take time installing insightface. Do not rerun NB23.

In [ ]:
# pyiqa 0.1.13 requires transformers==4.37.2; frozen N07 FER uses transformers==4.44.2.
# Preserve FER 4.44.2; install pyiqa without its incompatible dependency pin.
%pip -q install 'numpy==1.26.4' 'pandas>=2.2,<3' 'Pillow>=10,<12' 'torch>=2.2' 'transformers==4.44.2' 'huggingface-hub==0.36.2' 'onnxruntime==1.20.1' 'opencv-python-headless==4.10.0.84' 'insightface==0.7.3' 'scipy>=1.12,<2' 'timm>=0.8' 'einops' 'addict' 'facexlib' 'openai-clip' 'lmdb' 'pyyaml' 'scikit-image' 'sentencepiece' 'accelerate' 'icecream' 'future' 'yapf'
%pip -q install --no-deps 'pyiqa==0.1.13'
# A prior Colab pip operation may leave a stale transformers package loaded in memory.
# Verify the pinned version and its internal export before loading the ViT models.
import importlib.metadata as _md
assert _md.version('transformers') == '4.44.2', 'STOP: installed transformers version is not frozen 4.44.2'
# If a previous failed import occurred in this session, restart runtime once after successful installs.
from google.colab import drive
drive.mount('/content/drive')
import json, hashlib, math, os
from pathlib import Path
import numpy as np, pandas as pd, torch, cv2
from PIL import Image
from transformers import AutoImageProcessor,AutoModelForImageClassification
from insightface.model_zoo import get_model
from insightface.utils import face_align
import pyiqa
ROOT=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=ROOT/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N23=ROOT/'nb23_final_heldout_regeneration_frozen_v1'
OUT=ROOT/'nb24_unified_heldout_evaluation_frozen_v1'
OUT.mkdir(parents=True,exist_ok=True)
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
 return h.hexdigest()
def atomic_json(path,obj):
 tmp=path.with_suffix(path.suffix+'.partial')
 tmp.write_text(json.dumps(obj,indent=2,allow_nan=False))
 os.replace(tmp,path)
print('NB24 output:',OUT)

In [ ]:
status=json.loads((N23/'metadata/status.json').read_text())
assert status['expected']==status['verified']==324 and status['status']=='GENERATED_COMPLETE_EVALUATION_PENDING',status
protocol=json.loads((N23/'metadata/private_frozen_protocol.json').read_text())
assert len(protocol['jobs'])==324 and len(set(j['name'] for j in protocol['jobs']))==324
assert protocol['flowedit_commit']=='85594626161caca9dafa68592c1ffa21db87a806'
assert protocol['sd3_revision']=='ea42f8cef0f178587cf766dc8129abd379c90671'
assert protocol['excluded_tuning']==sorted(protocol['excluded_tuning']) and len(protocol['excluded_tuning'])==12
jobs=pd.DataFrame(protocol['jobs'])
assert jobs.source_id.nunique()==108 and set(jobs.condition)=={'n07_g4p5','n07_g5p5','same_expression_g4p5'}
assert jobs.groupby('source_id').size().eq(3).all() and set(jobs.seed)=={42}
rows=[]
for j in protocol['jobs']:
 name=j['name'];p=N23/'images'/f'{name}.png';r=N23/'images'/f'{name}.receipt.json'
 assert p.is_file() and r.is_file(),f'Missing NB23 output/receipt: {name}'
 receipt=json.loads(r.read_text())
 assert receipt['status']=='VERIFIED' and sha(p)==receipt['output_sha256'] and receipt['source_sha256']==j['source_sha256']
 assert receipt['runner_commit']==protocol['flowedit_commit'] and receipt['sd3_revision']==protocol['sd3_revision']
 with Image.open(p) as im: assert im.size==(512,512)
 src=Path(j['source_path'])
 assert src.is_file() and src.is_relative_to(N07/'sources') and sha(src)==j['source_sha256']
 rows.append({**j,'image_path':str(p),'image_sha256':receipt['output_sha256']})
df=pd.DataFrame(rows).sort_values(['source_id','condition']).reset_index(drop=True)
assert len(df)==324 and not df.duplicated(['source_id','condition']).any()
selected=pd.read_csv(N07/'selected_sources.csv')
assert len(selected)==120 and selected.source_id.is_unique
assert set(df.source_id)==set(selected.source_id)-set(protocol['excluded_tuning'])
raw=pd.read_csv(N07/'raw_results.csv')
assert len(raw)==600 and not raw.duplicated(['source_id','condition','seed']).any()
resolved=json.loads((N07/'resolved_config.json').read_text())
print('PREFLIGHT PASS — 108 held-out sources, 324 SHA-verified images, 0 tuning sources')

In [ ]:
MODELS={'trpakov/vit-face-expression':'ef0bc6fc34241b6587e7e009e7711357be28c024',
'mo-thecreator/vit-Facial-Expression-Recognition':'5aae4f09e1da247eb0f1c1e6f53271601fcafe09'}
assert all(resolved['evaluators']['emotion_model_revisions'][k]==v for k,v in MODELS.items())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
E={}
for ident,revision in MODELS.items():
 processor=AutoImageProcessor.from_pretrained(ident,revision=revision)
 model=AutoModelForImageClassification.from_pretrained(ident,revision=revision).to(DEVICE).eval()
 E[ident]=(processor,model)
 print('Loaded frozen FER:',ident)
CANON={'anger':'angry','happiness':'happy','sadness':'sad','surprised':'surprise','fearful':'fear','neutrality':'neutral','disgusted':'disgust'}
def canonical(v):
 v=str(v).strip().lower()
 return CANON.get(v,v)
def fer_eval(path):
 with Image.open(path) as handle: img=handle.convert('RGB')
 dists={}
 for ident,(processor,model) in E.items():
  inputs=processor(images=img,return_tensors='pt').to(DEVICE)
  with torch.inference_mode(): probs=torch.softmax(model(**inputs).logits,dim=-1)[0]
  dist={}
  for i,p in enumerate(probs):
   label=canonical(model.config.id2label[i]);dist[label]=dist.get(label,0.)+float(p.item())
  dists[ident]=dist
 labels=sorted(set().union(*(set(d) for d in dists.values())))
 ensemble={k:float(np.mean([d.get(k,0.) for d in dists.values()])) for k in labels}
 assert abs(sum(ensemble.values())-1)<1e-4
 return max(ensemble,key=ensemble.get),ensemble
# Frozen original N07 replay gate on ALL 108 held-out sources (not guessed baseline).
# N07 raw_results.csv uses 'original_predicted_class' across repeated generated rows.
assert {'source_id','original_predicted_class'}.issubset(raw.columns)
source_ref=raw[['source_id','original_predicted_class']].drop_duplicates()
assert not source_ref.duplicated('source_id').any(), 'STOP: inconsistent original predictions across N07 rows'
assert set(df.source_id).issubset(set(source_ref.source_id))
source_ref=source_ref.rename(columns={'original_predicted_class':'original_fer_predicted_class'})
old=dict(zip(source_ref.source_id,source_ref.original_fer_predicted_class))
orig_rows=[]
for i,sid in enumerate(sorted(df.source_id.unique()),1):
 p=N07/'sources'/f'{sid}.png'
 pred,dist=fer_eval(p)
 orig_rows.append(dict(source_id=sid,original_fer_predicted_class=pred,
                  original_emotion_probabilities_json=json.dumps(dist,sort_keys=True),
                  historical_fer_predicted_class=canonical(old[sid]),replay_equal=(pred==canonical(old[sid]))))
 if i%25==0:print('FER source replay',i,'/108',flush=True)
orig=pd.DataFrame(orig_rows)
orig.to_csv(OUT/'private_original_fer_replay.csv',index=False)
print('N07 FER replay:',int(orig.replay_equal.sum()),'/108')
assert orig.replay_equal.all(),'STOP: N07 original FER replay mismatch; no target evaluation permitted'

In [ ]:
fer_path=OUT/'private_target_fer_rows.csv'
prior=pd.read_csv(fer_path) if fer_path.exists() else pd.DataFrame()
if len(prior): assert not prior.duplicated('name').any()
fer_rows=[]
for i,r in enumerate(df.itertuples(index=False),1):
 hit=prior[(prior.name==r.name)&(prior.image_sha256==r.image_sha256)] if len(prior) else pd.DataFrame()
 if len(hit)==1: fer_rows.append(hit.iloc[0].to_dict())
 else:
  pred,dist=fer_eval(r.image_path)
  fer_rows.append(dict(name=r.name,source_id=r.source_id,condition=r.condition,
      image_sha256=r.image_sha256,fer_predicted_class=pred,
      fer_probabilities_json=json.dumps(dist,sort_keys=True)))
  pd.DataFrame(fer_rows).to_csv(fer_path,index=False)
 if i%25==0:print('FER target',i,'/324',flush=True)
fer=pd.DataFrame(fer_rows)
assert len(fer)==324 and not fer.duplicated('name').any()
evaldf=df.merge(fer[['name','image_sha256','fer_predicted_class']],on=['name','image_sha256'],validate='one_to_one')
evaldf=evaldf.merge(orig[['source_id','original_fer_predicted_class']],on='source_id',validate='many_to_one')
evaldf['fer_matches_human_label']=evaldf.fer_predicted_class==evaldf.emotion.map(canonical)
evaldf['fer_matches_original_prediction']=evaldf.fer_predicted_class==evaldf.original_fer_predicted_class
print(evaldf.groupby('condition')[['fer_matches_human_label','fer_matches_original_prediction']].mean())

In [ ]:
# Frozen N12 V3 weights: consult *private existing* N09 model_hash manifest, never silently fetch other weights.
NB09=ROOT/'20261013_nb09_detector-rootcause_4fae269d_3a2979d8_20261013T201108057643Z_02a058fe'
NB09_SESSION=NB09/'sessions'/'20261013T201108057643Z_02a058fe'
manifest=NB09_SESSION/'model_file_hashes.json'
assert manifest.is_file(),'STOP: frozen N09 private ONNX manifest absent; do not substitute models'
archives=json.loads(manifest.read_text())
EXPECTED={'detector':'5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91',
 'arcface':'4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43',
 'auraface':'a7933ea5330113b01c9b60351d8f4c33003f145d8470ac5f0e52ee2effe25c60'}
paths={}
for name,expected in EXPECTED.items():
 matched=[Path(p) for p,h in archives.items() if h==expected and Path(p).is_file()]
 assert matched,f'Missing exact frozen {name} ONNX file'
 assert sha(matched[0])==expected
 paths[name]=matched[0]
detector=get_model(str(paths['detector']),providers=['CPUExecutionProvider'])
assert type(detector).__module__=='insightface.model_zoo.retinaface' and type(detector).__name__=='RetinaFace', 'STOP: unexpected InsightFace detector wrapper'
assert getattr(detector,'taskname',None)=='detection', 'STOP: ONNX is not a detector'
assert [x.name for x in detector.session.get_outputs()]==['448','471','494','451','474','497','454','477','500'], 'STOP: detector ONNX output signature changed'
# SHA256 above pins the exact det_10g.onnx; InsightFace 0.7.3 calls this SCRFD-layout detector RetinaFace.
detector.prepare(ctx_id=-1,input_size=(640,640),det_thresh=0.1)
recognizers={}
for name in ('arcface','auraface'):
 model=get_model(str(paths[name]),providers=['CPUExecutionProvider'])
 model.prepare(ctx_id=-1)
 recognizers[name]=model
def embeds(path):
 rgb=np.asarray(Image.open(path).convert('RGB'))
 assert rgb.shape==(512,512,3)
 bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
 canvas=np.zeros((640,640,3),dtype=np.uint8)
 canvas[160:480,160:480]=cv2.resize(bgr,(320,320),interpolation=cv2.INTER_AREA)
 boxes,landmarks=detector.detect(canvas,max_num=0,metric='default')
 if not len(boxes):return {},False
 ix=int(np.argmax(boxes[:,4]))
 pts=(np.asarray(landmarks[ix],dtype=np.float64)-160)/0.625
 xy=(np.asarray(boxes[ix][:4],dtype=np.float64)-160)/0.625
 x1,y1,x2,y2=xy
 area=max(0,x2-x1)*max(0,y2-y1)
 overlap=max(0,min(512,x2)-max(0,x1))*max(0,min(512,y2)-max(0,y1))
 plausible=bool(pts.shape==(5,2) and np.isfinite(pts).all() and np.linalg.norm(pts[0]-pts[1])>=1 and (overlap/area if area else 0)>=0.7)
 aligned=face_align.norm_crop(bgr,landmark=pts.astype(np.float32),image_size=112)
 plausible=plausible and aligned.shape==(112,112,3) and aligned.dtype==np.uint8
 out={}
 if plausible:
  for name,model in recognizers.items():
   v=np.asarray(model.get_feat(aligned),dtype=np.float32).reshape(-1)
   norm=float(np.linalg.norm(v))
   if v.size==512 and np.isfinite(v).all() and np.isfinite(norm) and norm>1e-10:out[name]=v/norm
 return out,bool(plausible)
print('Frozen biometric ONNX hashes verified, SCRFD V3 ready')

In [ ]:
biom_path=OUT/'private_biometrics_rows.csv'
biom_prior=pd.read_csv(biom_path) if biom_path.exists() else pd.DataFrame()
if len(biom_prior):assert not biom_prior.duplicated('name').any()
src_embeddings={}
for i,sid in enumerate(sorted(df.source_id.unique()),1):
 src=N07/'sources'/f'{sid}.png'
 src_embeddings[sid]=embeds(src)
 if i%25==0:print('Biometric originals',i,'/108',flush=True)
biom_rows=[]
for i,r in enumerate(df.itertuples(index=False),1):
 hit=biom_prior[(biom_prior.name==r.name)&(biom_prior.image_sha256==r.image_sha256)] if len(biom_prior) else pd.DataFrame()
 if len(hit)==1:biom_rows.append(hit.iloc[0].to_dict())
 else:
  dst,ok=embeds(r.image_path)
  original,src_ok=src_embeddings[r.source_id]
  b=dict(name=r.name,source_id=r.source_id,condition=r.condition,image_sha256=r.image_sha256,
         source_detection_valid=src_ok,output_detection_valid=ok,manual_face_qc='PENDING')
  for model in ('arcface','auraface'):
   valid=model in original and model in dst
   b[model+'_pair_valid']=bool(valid)
   b[model+'_cosine']=float(np.dot(original[model],dst[model])) if valid else np.nan
  biom_rows.append(b)
  pd.DataFrame(biom_rows).to_csv(biom_path,index=False)
 if i%25==0:print('Biometrics',i,'/324',flush=True)
biom=pd.DataFrame(biom_rows)
assert len(biom)==324 and not biom.duplicated('name').any()
evaldf=evaldf.merge(biom[['name','image_sha256','source_detection_valid','output_detection_valid','arcface_pair_valid','arcface_cosine','auraface_pair_valid','auraface_cosine','manual_face_qc']],on=['name','image_sha256'],validate='one_to_one')
print(evaldf.groupby('condition')[['arcface_pair_valid','auraface_pair_valid']].mean())

In [ ]:
niqe_path=OUT/'private_niqe_rows.csv'
niqe_prior=pd.read_csv(niqe_path) if niqe_path.exists() else pd.DataFrame()
if len(niqe_prior):assert not niqe_prior.duplicated('name').any()
quality=pyiqa.create_metric('niqe',device='cpu')
quality.eval()
def niqe_score(path):
 rgb=np.asarray(Image.open(path).convert('RGB'),dtype=np.float32)/255.
 tensor=torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0)
 with torch.inference_mode(): val=float(quality(tensor).detach().cpu().item())
 assert np.isfinite(val),f'Non-finite NIQE: {path}'
 return val
src_niqe={sid:niqe_score(N07/'sources'/f'{sid}.png') for sid in sorted(df.source_id.unique())}
niqe_rows=[]
for i,r in enumerate(df.itertuples(index=False),1):
 hit=niqe_prior[(niqe_prior.name==r.name)&(niqe_prior.image_sha256==r.image_sha256)] if len(niqe_prior) else pd.DataFrame()
 if len(hit)==1:niqe_rows.append(hit.iloc[0].to_dict())
 else:
  niqe_rows.append(dict(name=r.name,source_id=r.source_id,condition=r.condition,
     image_sha256=r.image_sha256,niqe=niqe_score(r.image_path),original_niqe=src_niqe[r.source_id]))
  pd.DataFrame(niqe_rows).to_csv(niqe_path,index=False)
 if i%25==0:print('NIQE',i,'/324',flush=True)
nq=pd.DataFrame(niqe_rows)
assert len(nq)==324 and not nq.duplicated('name').any()
evaldf=evaldf.merge(nq[['name','image_sha256','niqe','original_niqe']],on=['name','image_sha256'],validate='one_to_one')
evaldf['delta_niqe_vs_source']=evaldf.niqe-evaldf.original_niqe
print(evaldf.groupby('condition')[['niqe','delta_niqe_vs_source']].mean())

In [ ]:
# Single independent unit = SOURCE (one seed per source per condition). Paired bootstrap resamples source IDs.
METRICS=['arcface_cosine','auraface_cosine','fer_matches_human_label','fer_matches_original_prediction','niqe','delta_niqe_vs_source']
evaldf.to_csv(OUT/'private_all_324_evaluation_rows.csv',index=False)
def bootstrap_ci(v,seed=24024,B=10000):
 arr=np.asarray(v,dtype=float)
 assert len(arr)==108 and np.isfinite(arr).all()
 rng=np.random.default_rng(seed)
 sample=arr[rng.integers(0,len(arr),(B,len(arr)))].mean(axis=1)
 return float(np.quantile(sample,.025)),float(np.quantile(sample,.975))
summary=[]
for cond,g in evaldf.groupby('condition'):
 assert len(g)==108 and g.source_id.nunique()==108
 for metric in METRICS:
  vals=g[metric].astype(float)
  finite=vals[np.isfinite(vals)]
  lo,hi=bootstrap_ci(finite.to_numpy()) if len(finite)==108 else (np.nan,np.nan)
  summary.append(dict(condition=cond,metric=metric,n_sources=108,n_valid=len(finite),
                      mean=float(finite.mean()) if len(finite) else np.nan,
                      ci95_low=lo,ci95_high=hi,
                      ci_status='complete' if len(finite)==108 else 'NOT_ESTIMATED_MISSING_PAIRS'))
summ=pd.DataFrame(summary);summ.to_csv(OUT/'aggregate_by_condition.csv',index=False)
pairs=[]
for metric in METRICS:
 pivot=evaldf.pivot(index='source_id',columns='condition',values=metric)
 for a,b in [('n07_g5p5','n07_g4p5'),('same_expression_g4p5','n07_g4p5'),('same_expression_g4p5','n07_g5p5')]:
  delta=(pivot[a].astype(float)-pivot[b].astype(float)).dropna()
  lo,hi=bootstrap_ci(delta.to_numpy()) if len(delta)==108 else (np.nan,np.nan)
  pairs.append(dict(metric=metric,contrast=f'{a} minus {b}',n_paired_sources=len(delta),mean_delta=float(delta.mean()) if len(delta) else np.nan,
                    ci95_low=lo,ci95_high=hi,
                    ci_status='complete' if len(delta)==108 else 'NOT_ESTIMATED_MISSING_PAIRS'))
paired=pd.DataFrame(pairs);paired.to_csv(OUT/'paired_contrasts.csv',index=False)
assert len(evaldf)==324
complete_biom=bool(evaldf[['arcface_pair_valid','auraface_pair_valid']].all().all())
final_status={'status':'EVALUATION_COMPLETE_QC_PENDING' if complete_biom else 'EVALUATION_PARTIAL_BIOMETRIC_MISSING_QC_PENDING',
  'generated_verified':324,'evaluated_rows':324,'n_sources':108,'conditions':3,
  'biometric_arcface_valid':int(evaldf.arcface_pair_valid.sum()),
  'biometric_auraface_valid':int(evaldf.auraface_pair_valid.sum()),
  'original_fer_replay':int(orig.replay_equal.sum()),
  'niqe_valid':int(evaldf.niqe.notna().sum()),
  'manual_qc':'PENDING','validated_privacy_claim':False,
  'interpretation':'Biometric cosine is technical similarity, not verified re-identification risk. NIQE is naturalness, not clinical fidelity.'}
atomic_json(OUT/'status.json',final_status)
display(summ.round(4));display(paired.round(4))
print('NB24 COMPLETE:',json.dumps(final_status,indent=2))